# saves term — E[saves] one gameweek ahead (GK)

**Render-not-decide.** A re-runnable view of the `saves` term, **not** the record — the frozen numbers live in [predictive-phase2-component-model.md](../../../docs/studies/results/predictive-phase2-component-model.md). Logic is in `model/terms/saves/` (+ the shared Poisson-player base).

GK only (~18% of GK points). GK ranking is near-chance for every model — saves lift GK to **parity**, an honest ceiling (see `ASSUMPTIONS.md`). Population: `position==GK`, `minutes > 0`, DGW excluded, GW > 3.

## Setup

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart
from model.terms.saves import SavesModel, SavesTerm

mart = load_mart()
term = SavesTerm(SavesModel(variant="minimal"))
term.model.pool.minimal, [f.name for f in term.model.pool.candidates]

## Pre-fit assumptions
> Is Poisson justified (saves may be over-dispersed — reported, not blocked) and is the effect learnable at this N? See `ASSUMPTIONS.md` §3.

In [ ]:
report = term.model.check_assumptions(SavesModel.population(mart))
print(f"family_ok={report.family_ok}  detectable={report.detectable}  n_train={report.n_train}")
report.dispersion

## Gate — E[saves] vs the keeper's own lagged-saves baseline
> Per-term level (spec §5): does the xGC-based model out-rank a keeper's naive saves history? GK only; the ceiling is low.

In [ ]:
gate = term.validate(mart)
display(gate.table)

ax = gate.table.set_index("position")[["baseline", "e_saves"]].plot.bar(figsize=(5, 3.5))
ax.set_ylabel("within-position Spearman"); ax.set_title("saves: model vs own baseline (GK)"); plt.tight_layout()

## Diagnose — residuals + ablation
> Post-gate. Which keeper-GWs the model misses, and the measured contribution of `xgc_roll3` vs `minutes_roll3`.

In [ ]:
diag = term.diagnose(mart)
display(diag.ablation)
display(diag.residuals.head(10))